# 03 — Direct Mode: Potentiostat Control

**Direct mode** lets you control the potentiostat manually — setting potential or current and reading measurements instantly — without loading a method file.

Use direct mode for:
- Quick spot measurements (OCV, hold potential)
- Conditioning or equilibration steps
- Troubleshooting a device or cell setup
- Building custom measurement loops in Python

For automated experiment sequences (LSV, CV, EIS…) see **`06_method_mode.ipynb`**.

### Prerequisites
- IviumSoft running
- Device connected (see `02_device_and_instance_management.ipynb`)

In [ ]:
import time
import matplotlib.pyplot as plt
from pyvium import Pyvium
from pyvium.errors import CellOffError, IllegalCommandError, InvalidStateError

Pyvium.open_driver()
try:
    Pyvium.connect_device()
    print("Ready")
except Exception as e:
    print(f"Could not connect: {e}")

## 1. Connection Modes

The connection mode determines the electrode wiring. Always set this before turning the cell on.

| Code | Mode | Use case |
|------|------|----------|
| 0 | Off | No connection |
| 1 | EStat 4EL | Standard 4-electrode potentiostat (default) |
| 2 | EStat 2EL | 2-electrode potentiostat |
| 3–6 | EStat Dummy | Dummy cell (testing/calibration) |
| 7 | IStat 4EL | 4-electrode galvanostat |
| 8 | IStat 2EL | 2-electrode galvanostat |
| 9 | IStat Dummy | Galvanostat dummy cell |
| 10 | BiStat 4EL | Bipotentiostat 4-electrode |
| 11 | BiStat 2EL | Bipotentiostat 2-electrode |

In [ ]:
# Set to standard 4-electrode potentiostat
Pyvium.set_connection_mode(1)
print("Connection mode: EStat 4EL (standard potentiostat)")

## 2. Cell Status

`get_cell_status()` returns a list of active status flags decoded from a bitmask. The flags and their corresponding bits:

| Flag | Bit | Meaning |
|------|-----|---------|
| `'Cell on'` | — | Cell output is active |
| `'Cell off'` | — | Cell output is disabled (default after connect) |
| `'I_ovl'` | 2 | Current overload — reduce the current range |
| `'Anin1_ovl'` | 4 | Analog input 1 overload |
| `'E_ovl'` | 5 | Potential overload — reduce the potential range |
| `'CellOff_button pressed'` | 7 | Hardware safety button was pressed |

When `'I_ovl'` or `'E_ovl'` is active the measurement is unreliable — check your current range and applied potential.

> **Important:** Direct mode settings (potential, current range, filter, stability) are **suspended** while a method is running via `start_method()`. They resume when the method finishes or is aborted. Configure direct mode settings only when no method is active.

In [ ]:
status = Pyvium.get_cell_status()
print(f"Cell status flags: {status}")

# Check for overloads before trusting measurements
OVERLOAD_FLAGS = {'I_ovl', 'E_ovl', 'Anin1_ovl'}
overloads = OVERLOAD_FLAGS & set(status)
if overloads:
    print(f"WARNING: overload detected: {overloads}")
else:
    print("No overloads — measurements are reliable")

## 3. Current Ranges

Set the current range **before** turning the cell on. A range that is too large gives poor resolution; one that is too small will cause an overload.

The DLL defines current range codes as a descending decade sequence. According to the IviumSoft DLL reference, `IV_setcurrentrange` starts at **0 = 10 A, 1 = 1 A**, and continues down in decades:

| Code | Range (IviumStat / full-range devices) |
|------|----------------------------------------|
| 0 | 10 A |
| 1 | 1 A |
| 2 | 100 mA |
| 3 | 10 mA |
| 4 | 1 mA |
| 5 | 100 µA |
| 6 | 10 µA |
| 7 | 1 µA |
| 8 | 100 nA |
| … | … |

Low-current devices (CompactStat, pocketSTAT) only support a subset of the lower ranges — the codes still follow the same descending decade pattern but the device's maximum range is lower than 10 A, so code 0 maps to whatever the highest available range is for that model.

> **Always verify the correct code for your device** by looking at IviumSoft's direct-mode current range dropdown before using these values in a script.

> For WE2 (BiStat), the scale is different: `IV_setcurrentrangeWE2` starts at 0 = 10 mA, 1 = 1 mA, etc. See notebook `05_bipotentiostat_and_we32.ipynb`.

In [ ]:
Pyvium.set_current_range(2)  # 100 µA range
print("Current range set to 100 µA")

## 4. Filter and Stability Settings

**Filter** controls the bandwidth of the current measurement. Lower bandwidth = less noise but slower response.

| Code | Filter |
|------|--------|
| 0 | 1 MHz (fastest, most noise) |
| 1 | 100 kHz |
| 2 | 10 kHz |
| 3 | 1 kHz |
| 4 | 10 Hz (slowest, least noise) |

**Stability** adjusts the potentiostat feedback loop for the cell impedance:

| Code | Stability | Use when |
|------|-----------|----------|
| 0 | High Speed | Low-impedance cells, fast scans |
| 1 | Standard | Most electrochemical cells (default) |
| 2 | High Stability | High-impedance cells, noisy signals |

In [ ]:
Pyvium.set_filter(3)       # 1 kHz filter
Pyvium.set_stability(1)    # Standard stability
print("Filter: 1 kHz | Stability: Standard")

## 5. Turning the Cell On and Setting Potential

> **Safety note:** Always set the desired potential **before** turning the cell on. This prevents a transient spike to an unexpected value.

In [ ]:
# Set potential BEFORE turning cell on
Pyvium.set_potential(0.0)  # 0 V vs reference
print("Target potential: 0.0 V")

Pyvium.set_cell_on()
status = Pyvium.get_cell_status()
print(f"Cell status: {status}")

## 6. Reading Potential and Current

After the cell is on, read potential and current. Give the potentiostat a short settling time after changing potential.

In [ ]:
time.sleep(0.1)  # allow settling

potential = Pyvium.get_potential()
current   = Pyvium.get_current()

print(f"Measured potential : {potential:.6f} V")
print(f"Measured current   : {current:.3e} A")

## 7. Potentiostatic Hold — Sampling Over Time

A common pattern: hold the potential fixed and record current vs. time (chronoamperometry).

In [ ]:
HOLD_POTENTIAL = 0.2   # V
DURATION_S     = 5.0   # seconds
INTERVAL_S     = 0.2   # sampling interval

Pyvium.set_potential(HOLD_POTENTIAL)
Pyvium.set_cell_on()
time.sleep(0.1)  # settling

timestamps = []
currents   = []
t_start    = time.time()

while (time.time() - t_start) < DURATION_S:
    t = time.time() - t_start
    I = Pyvium.get_current()
    timestamps.append(t)
    currents.append(I * 1e6)  # convert to µA
    time.sleep(INTERVAL_S)

Pyvium.set_cell_off()
print(f"Collected {len(timestamps)} points over {DURATION_S:.0f} s")

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(timestamps, currents, 'b-o', markersize=3)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Current (µA)")
ax.set_title(f"Chronoamperometry — Hold at {HOLD_POTENTIAL} V")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Galvanostatic Mode (Set Current)

In galvanostatic mode the instrument controls current and you read potential. Switch to an IStat connection mode first.

In [ ]:
HOLD_CURRENT = 1e-6  # 1 µA

Pyvium.set_connection_mode(7)   # IStat 4EL (galvanostat)
Pyvium.set_current_range(4)     # 1 µA range
Pyvium.set_current(HOLD_CURRENT)
Pyvium.set_cell_on()
time.sleep(0.1)

potential = Pyvium.get_potential()
current   = Pyvium.get_current()
print(f"Applied current  : {current:.3e} A")
print(f"Measured potential: {potential:.6f} V")

Pyvium.set_cell_off()
Pyvium.set_connection_mode(1)   # back to potentiostat

## 9. Open Circuit Potential (OCV)

OCV is measured with the cell **off** — the instrument only listens, it does not apply any signal.

In [ ]:
OCV_DURATION_S = 10.0
INTERVAL_S     = 0.5

Pyvium.set_cell_off()  # cell must be off for true OCV

timestamps = []
potentials = []
t_start    = time.time()

while (time.time() - t_start) < OCV_DURATION_S:
    t = time.time() - t_start
    E = Pyvium.get_potential()
    timestamps.append(t)
    potentials.append(E)
    time.sleep(INTERVAL_S)

print(f"OCV at end: {potentials[-1]:.4f} V")

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(timestamps, potentials, 'g-')
ax.set_xlabel("Time (s)")
ax.set_ylabel("OCV (V)")
ax.set_title("Open Circuit Potential vs. Time")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cleanup

In [ ]:
try:
    Pyvium.set_cell_off()
    Pyvium.disconnect_device()
    Pyvium.close_driver()
except Exception as e:
    print(f"Cleanup error: {e}")
print("Done")

---

## Summary

| Task | Method |
|------|--------|
| Set electrode wiring | `set_connection_mode(n)` |
| Check cell state | `get_cell_status()` |
| Set current range | `set_current_range(n)` |
| Set measurement filter | `set_filter(n)` |
| Set stability | `set_stability(n)` |
| Apply potential (potentiostat) | `set_potential(V)` |
| Apply current (galvanostat) | `set_current(A)` |
| Read potential | `get_potential()` → float |
| Read current | `get_current()` → float |
| Enable / disable cell | `set_cell_on()` / `set_cell_off()` |

## Next

- **`04_direct_mode_signals.ipynb`** — High-speed traces, DAC/ADC, digital I/O, AC control
- **`05_bipotentiostat_and_we32.ipynb`** — BiStat (WE2) and 32-channel WE32 array
- **`06_method_mode.ipynb`** — Run full electrochemical methods (LSV, CV, EIS…)